# DiffuScene Setup on Google Colab
Notebook này thiết lập môi trường và tải sẵn các pretrained model, dataset đã được xử lý (preprocessed) để không cần phải chạy lại các bước tốn thời gian.

In [1]:
# Clone source code (Bỏ comment phần này nếu bạn upload notebook lên một Colab trống)
!git clone https://github.com/AdamHermes/DiffuScene.git
%cd DiffuScene


Cloning into 'DiffuScene'...
remote: Enumerating objects: 319, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 319 (delta 139), reused 113 (delta 102), pack-reused 119 (from 1)
Receiving objects: 100% (319/319), 1.53 MiB | 22.74 MiB/s, done.
Resolving deltas: 100% (179/179), done.
/content/DiffuScene


In [ ]:
%%bash
# Install Miniconda silently
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -q -O miniconda.sh
bash miniconda.sh -b -f -p /usr/local > /dev/null 2>&1
source /usr/local/etc/profile.d/conda.sh

# Accept Anaconda Terms of Service (Required!)
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Create environment from YAML
cd /content/DiffuScene
conda env create -f environment.yaml


In [ ]:
%%bash
cd /content/DiffuScene
source /usr/local/etc/profile.d/conda.sh
conda activate diffuscene

python setup.py build_ext --inplace
pip install -e .

cd ChamferDistancePytorch/chamfer3D
python setup.py install


## Tải Pretrained Models và Preprocessed Datasets
Sử dụng `gdown` để tải trực tiếp từ Google Drive của tác giả.

In [4]:
!pip install --upgrade gdown

import os
os.makedirs("models", exist_ok=True)
os.makedirs("datasets", exist_ok=True)

# 1. Pretrained models of DiffuScene
!gdown 1pk9AzGcBz_kRfmRzvFNDW5byk4MwbXEm -O models/pretrained_models.zip


# 3. Preprocessed 3D-Front dataset
!gdown 1UNSFN0kULyOzUErDPVvkKYbmfzA-4MsG -O datasets/3D-Front_preprocessed.zip


# Giải nén các file (Có thể mất một lúc tùy vào kích thước)
!unzip -q models/pretrained_models.zip -d models/
!unzip -q datasets/3D-Front_preprocessed.zip -d datasets/


DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/chamfer_3D-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --id
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized argume

## Chạy Evaluation
Môi trường và dữ liệu đã sẵn sàng. Bạn có thể sử dụng `%%bash` cell như dưới đây để thực hiện eval/generate. (Cần kiểm tra lại các script shell `generate.sh` để trỏ đúng đường dẫn dataset/models bạn vừa giải nén nếu mặc định không khớp).

In [5]:
%%bash

# Chạy ví dụ generate (Mở uncomment để chạy)
# ./run/generate.sh
# ./run/generate_text.sh

In [7]:
# Kết nối Google Drive để lưu kết quả
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
%%writefile scripts/generate_bboxes.py
import argparse
import os
import sys
import numpy as np
import torch
import json
import glob

from training_utils import load_config
from scene_synthesis.datasets import filter_function, get_dataset_raw_and_encoded
from scene_synthesis.networks import build_network

def main(argv):
    parser = argparse.ArgumentParser(description="Generate bounding boxes using a previously trained model")
    parser.add_argument("config_file", help="Path to experiment config")
    parser.add_argument("output_directory", default="/tmp/", help="Path to output directory")
    parser.add_argument("--weight_file", default=None, help="Path to a pretrained model")
    parser.add_argument("--n_sequences", default=10, type=int, help="Number of sequences to generate")
    parser.add_argument("--scene_id", default=None, help="The scene id to be used for conditioning")
    parser.add_argument("--clip_denoised", action="store_true", help="if clip_denoised")
    parser.add_argument("--fix_order", action="store_true", help="if use fix order")
    args = parser.parse_args(argv)

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("Running code on", device)

    if not os.path.exists(args.output_directory):
        os.makedirs(args.output_directory)

    config = load_config(args.config_file)
    
    # 1. Dynamically find the dataset directory
    stats_files = glob.glob("../datasets/**/dataset_stats.txt", recursive=True)
    target_dir = None
    for f in stats_files:
        if "bedrooms" in f:
            target_dir = os.path.dirname(f)
            break
            
    if target_dir is None:
        raise FileNotFoundError("Could not find dataset_stats.txt for bedrooms in ../datasets/")
        
    config["data"]["dataset_directory"] = target_dir
    if "train_stats_file" in config["network"].get("diffusion_kwargs", {}):
        config["network"]["diffusion_kwargs"]["train_stats_file"] = os.path.join(target_dir, "dataset_stats.txt")

    # 2. Dynamically find threed_front.pkl
    pkl_files = glob.glob("../datasets/**/threed_front.pkl", recursive=True)
    if pkl_files:
        os.environ["PATH_TO_SCENES"] = pkl_files[0]
        print(f"Set PATH_TO_SCENES to {pkl_files[0]}")
    else:
        print("Warning: threed_front.pkl not found (not needed for cached dataset)")
        
    if "text" in config["data"]["encoding_type"] and "textfix" not in config["data"]["encoding_type"]:
        config["data"]["encoding_type"] = config["data"]["encoding_type"].replace("text", "textfix")
    if "no_prm" not in config["data"]["encoding_type"]:
        config["data"]["encoding_type"] += "_no_prm"

    raw_dataset, dataset = get_dataset_raw_and_encoded(
        config["data"],
        filter_fn=filter_function(config["data"], split=config["validation"].get("splits", ["test"])),
        split=config["validation"].get("splits", ["test"])
    )
    print("Loaded {} scenes with {} object types".format(len(dataset), dataset.n_object_types))
    
    network, _, _ = build_network(
        dataset.feature_size, dataset.n_classes,
        config, args.weight_file, device=device
    )
    network.eval()

    given_scene_id = None
    if args.scene_id:
        for i, di in enumerate(raw_dataset):
            if str(di.scene_id) == args.scene_id:
                given_scene_id = i

    export_data = {
        "scene_ids": [],
        "class_labels": [],
        "translations": [],
        "sizes": [],
        "angles": []
    }

    for i in range(args.n_sequences):
        scene_idx = given_scene_id or (i if args.fix_order and i < len(dataset) else (i % len(dataset) if args.fix_order else np.random.choice(len(dataset))))
        current_scene = raw_dataset[scene_idx]
        samples = dataset[scene_idx]
        print("{} / {}: Generating layout for scene {}".format(i, args.n_sequences, current_scene.scene_id))

        room_mask = current_scene.room_mask.unsqueeze(0).unsqueeze(0) if hasattr(current_scene, "room_mask") else None
        
        # Simplified generation
        bbox_params = network.generate_layout(
            room_mask=room_mask.to(device) if room_mask is not None else None,
            num_points=config["network"]["sample_num_points"],
            point_dim=config["network"]["point_dim"],
            text=samples["description"] if "description" in samples.keys() else None,
            device=device,
            clip_denoised=args.clip_denoised,
            batch_seeds=torch.arange(i, i+1),
        )

        boxes = dataset.post_process(bbox_params)
        
        unique_scene_id = "{}_{}_{:03d}".format(current_scene.scene_id, scene_idx, i)
        export_data["scene_ids"].append(unique_scene_id)
        export_data["class_labels"].append(boxes["class_labels"][0].cpu().numpy().tolist())
        export_data["translations"].append(boxes["translations"][0].cpu().numpy().tolist())
        export_data["sizes"].append(boxes["sizes"][0].cpu().numpy().tolist())
        export_data["angles"].append(boxes["angles"][0].cpu().numpy().tolist())

    json_path = os.path.join(args.output_directory, "collision_params.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(export_data, f, indent=2)
    print(f"Saved collision parameters to {json_path}")

if __name__ == "__main__":
    main(sys.argv[1:])


Overwriting scripts/generate_bboxes.py


In [12]:
!pip install torchtext

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/chamfer_3D-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.4 MB/s eta 0:00:00


In [13]:
%%bash

source /usr/local/etc/profile.d/conda.sh
conda activate diffuscene

cd scripts

# Instantly comment out the broken torchtext import using Linux sed
sed -i "s/import torchtext/# import torchtext/g" ../scene_synthesis/datasets/threed_front_dataset.py

# Download the missing NLTK dictionary that DiffuScene expects
python -c "import nltk; nltk.download(\"cmudict\")"

# Set parameters for unconditional bedrooms generation
config="../config/uncond/diffusion_bedrooms_instancond_lat32_v.yaml"
exp_name="bedrooms_uncond"
weight_file=$(find ../models -name "model_30000" | grep "${exp_name}" | head -n 1)
output_dir="/content/drive/MyDrive/DiffuScene_Results"

mkdir -p "${output_dir}"
echo "Running generation and saving bounding boxes to Google Drive..."
python generate_bboxes.py "${config}" "${output_dir}" --weight_file "${weight_file}" --n_sequences 20 --fix_order --clip_denoised
echo "Finished! Results saved to ${output_dir}/collision_params.json"


Running generation and saving bounding boxes to Google Drive...
Finished! Results saved to /content/drive/MyDrive/DiffuScene_Results/collision_params.json


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/_ops.py", line 1503, in load_library
    ctypes.CDLL(path)
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib/python3.12/dist-packages/torchtext/lib/libtorchtext.so: undefined symbol: _ZN5torch6detail10class_baseC2ERKSsS3_SsRKSt9type_infoS6_

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/DiffuScene/scripts/generate_bboxes.py", line 9, in <module>
    from scene_synthesis.datasets import filter_function, get_dataset_raw_and_encoded
  File "/content/DiffuScene/scene_synthesis/datasets/__init__.py", line 6, in <module>
    from .threed_front_dataset import dataset_encoding_factory
  File "/content/DiffuScene/scene_synthesis/datasets/threed_front_dataset.py", line 620, in <module>
    import torc